In [1]:
from active_testing_benchmark.modeling.preparation import prepare_data
from active_testing_benchmark.modeling.models import create_model
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import pandas as pd

from sklearn.metrics import roc_auc_score

2026-11-05 13:50:24.434 | INFO     | active_testing_benchmark.config:<module>:11 - PROJ_ROOT path is: /Users/alexandernielsen/Desktop/ITU/msc/thesis-data-distillation


In [2]:
data = prepare_data()
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = create_model(num_classes=len(data.classes), pretrained=True, device=device)

In [3]:
loss_fn = nn.CrossEntropyLoss(reduction="sum")
optimizer = torch.optim.Adam(params=model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)

In [4]:
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  print(f"Correct: {correct}, Total: {len(y_pred)}")
  acc = (correct / len(y_pred))
  return acc

In [5]:
def train_model(model, train_loader, loss_fn, optimizer):
  model.train()
  running_loss = 0

  y_preds = torch.tensor([]).to(device)
  ys = torch.tensor([]).to(device)

  for batch in tqdm(train_loader, desc="Training batches", position=1, leave=False):
    X, y = batch['image'].to(device), batch['target'].to(device)
    ys = torch.cat((ys, y), dim=0)
    y_pred = model(X)
    y_preds = torch.cat((y_preds, y_pred), dim=0)
    loss = loss_fn(y_pred, y)
    running_loss += loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  acc = accuracy_fn(ys, y_preds.argmax(dim=1))
  auc = roc_auc_score(ys.detach().cpu().numpy(), y_preds.detach()[:, 1].cpu().numpy())
  running_loss /= len(ys)
  return running_loss, acc, auc

In [6]:
def test_model(model, loss_fn, test_loader, scheduler):
  model.eval()
  test_loss = 0
  test_acc = 0

  ys = torch.tensor([]).to(device)
  y_preds = torch.tensor([]).to(device)

  with torch.inference_mode():
    for _, batch in enumerate(test_loader):
      X, y = batch['image'].to(device), batch['target'].to(device)
      ys = torch.cat((ys, y), dim=0)
      y_pred = model(X)
      y_preds = torch.cat((y_preds, y_pred), dim=0)
      loss = loss_fn(y_pred, y)
      test_loss += loss.item()

    test_acc = accuracy_fn(y_pred=y_preds.argmax(dim=1), y_true=ys)
    test_auc = roc_auc_score(ys.detach().cpu().numpy(), y_preds.detach()[:, 1].cpu().numpy())
    test_loss /= len(ys)
    scheduler.step(test_loss)

  return test_loss, test_acc, test_auc

In [7]:
train_loader = DataLoader(data.pilot_fit, batch_size=32, shuffle=True)
validation_loader = DataLoader(data.pilot_validation, batch_size=32, shuffle=False)

In [ ]:
epochs = 20
start = time.time()
rows = []
for epoch in tqdm(range(epochs), desc="Training epochs", position=1, leave=False):
  epoch_start = time.time()
  train_loss, train_acc, train_auc = train_model(model=model, loss_fn=loss_fn, optimizer=optimizer, train_loader=train_loader)
  validation_loss, validation_acc, validation_auc = test_model(model=model, loss_fn=loss_fn, test_loader=validation_loader, scheduler=scheduler)
  print(f"Epoch: {epoch} | Train loss: {train_loss:.3f} | Train acc {train_acc:.2%} | Train AUC {train_auc:.2%} | validation loss {validation_loss:.3f} | validation acc {validation_acc:.2%} | validation AUC {validation_auc:.2%}")
  epoch_end = time.time()

  row = {
    "Epoch": epoch,
    "Train Loss": train_loss,
    "Train Acc": train_acc,
    "Train AUC": train_auc,
    "validation Loss": validation_loss,
    "validation Acc": validation_acc,
    "validation AUC": validation_auc,
    "epoch time": epoch_end - epoch_start
  }

  rows.append(row)
training_results = pd.DataFrame(rows, ignore_index=True)
end = time.time()
print(f"Total time taken: {end-start}")

training_results

Training epochs:   0%|          | 0/20 [00:00<?, ?it/s]

Training batches:   0%|          | 0/150 [00:00<?, ?it/s]

Correct: 4220, Total: 4800
Correct: 1095, Total: 1200
Epoch: 0 | Train loss: 0.311 | Train acc 87.92% | Train AUC 72.39% | validation loss 0.257 | validation acc 91.25% | validation AUC 80.68%


Training batches:   0%|          | 0/150 [00:00<?, ?it/s]

Correct: 4404, Total: 4800
Correct: 1101, Total: 1200
Epoch: 1 | Train loss: 0.229 | Train acc 91.75% | Train AUC 84.68% | validation loss 0.278 | validation acc 91.75% | validation AUC 81.86%


Training batches:   0%|          | 0/150 [00:00<?, ?it/s]

... (32 outputs left out)

Correct: 4779, Total: 4800
Correct: 1103, Total: 1200
Epoch: 18 | Train loss: 0.020 | Train acc 99.56% | Train AUC 99.98% | validation loss 0.301 | validation acc 91.92% | validation AUC 84.14%


Training batches:   0%|          | 0/150 [00:00<?, ?it/s]

Correct: 4783, Total: 4800
Correct: 1108, Total: 1200
Epoch: 19 | Train loss: 0.020 | Train acc 99.65% | Train AUC 99.98% | validation loss 0.307 | validation acc 92.33% | validation AUC 84.24%


TypeError: DataFrame.__init__() got an unexpected keyword argument 'ignore_index'